In [2]:
import sqlite3
import pandas as pd

QUESTION 1 : which towns have the highest and lowest average resale prices?

ANSWER 1 : highest Bukit Timah, lowest Yishun



In [3]:
conn = sqlite3.connect("sg_dashboard.db")
df = pd.read_sql_query('''
                       SELECT ROUND(AVG(resale_price),0) AS avg_resale_price,
                       town
                       FROM hdb_resale
                       GROUP BY town 
                       ORDER BY avg_resale_price DESC
                       ''', conn)
df["avg_resale_price"] = df["avg_resale_price"].astype(int)
print(df)

    avg_resale_price             town
0             782616      BUKIT TIMAH
1             714623           BISHAN
2             696580     CENTRAL AREA
3             645138       QUEENSTOWN
4             642106      BUKIT MERAH
5             597892  KALLANG/WHAMPOA
6             590374        PASIR RIS
7             579004        TOA PAYOH
8             569044         TAMPINES
9             567839        SERANGOON
10            561536         CLEMENTI
11            559513    MARINE PARADE
12            547265          PUNGGOL
13            531636         SENGKANG
14            524648          HOUGANG
15            512299          GEYLANG
16            507716      BUKIT BATOK
17            505644    BUKIT PANJANG
18            505125        SEMBAWANG
19            493375    CHOA CHU KANG
20            485470       ANG MO KIO
21            484986            BEDOK
22            484165        WOODLANDS
23            476823      JURONG EAST
24            466029      JURONG WEST
25          

QUESTION 2 : how have prices in the West region trended over time compared to the island average?

ANSWER 2 : island increase more


In [6]:
west_towns = ('JURONG WEST', 'JURONG EAST', 'BUKIT BATOK', 'BUKIT PANJANG', 'CHOA CHU KANG')

df_west = pd.read_sql_query(f'''
                       SELECT ROUND(AVG(resale_price),0) AS avg_resale_price,
                       month
                       FROM hdb_resale
                       WHERE town IN {west_towns}
                       GROUP BY month 
                       ORDER BY month''', conn)
df_west["avg_resale_price"] = df_west["avg_resale_price"].astype(int)
print(df_west)

     avg_resale_price                month
0              395280  2017-01-01 00:00:00
1              392901  2017-02-01 00:00:00
2              398761  2017-03-01 00:00:00
3              393643  2017-04-01 00:00:00
4              402537  2017-05-01 00:00:00
..                ...                  ...
109            587911  2026-02-01 00:00:00
110            596030  2026-03-01 00:00:00
111            603381  2026-04-01 00:00:00
112            593363  2026-05-01 00:00:00
113            579320  2026-06-01 00:00:00

[114 rows x 2 columns]


In [7]:
df_island = pd.read_sql_query('''
                       SELECT ROUND(AVG(resale_price),0) AS avg_resale_price,
                       month
                       FROM hdb_resale
                       GROUP BY month 
                       ORDER BY month''', conn)
df_island["avg_resale_price"] = df_island["avg_resale_price"].astype(int)
print(df_island)

     avg_resale_price                month
0              427507  2017-01-01 00:00:00
1              447399  2017-02-01 00:00:00
2              445030  2017-03-01 00:00:00
3              438629  2017-04-01 00:00:00
4              443789  2017-05-01 00:00:00
..                ...                  ...
109            656734  2026-02-01 00:00:00
110            662156  2026-03-01 00:00:00
111            657876  2026-04-01 00:00:00
112            660422  2026-05-01 00:00:00
113            655782  2026-06-01 00:00:00

[114 rows x 2 columns]


In [8]:

df_west["west_indexed_%"] = ((df_west["avg_resale_price"]-df_west["avg_resale_price"].iloc[0])/ df_west["avg_resale_price"].iloc[0]) * 100
df_island["island_indexed_%"] = ((df_island["avg_resale_price"]-df_island["avg_resale_price"].iloc[0]) / df_island["avg_resale_price"].iloc[0]) * 100
df_indexed = df_west.merge(df_island, on='month', suffixes=("_west", "_island"))
df_indexed["island_lead_%"] = df_indexed["island_indexed_%"] - df_indexed["west_indexed_%"]
print(df_indexed[["month", "west_indexed_%", "island_indexed_%", "island_lead_%"]])

                   month  west_indexed_%  island_indexed_%  island_lead_%
0    2017-01-01 00:00:00        0.000000          0.000000       0.000000
1    2017-02-01 00:00:00       -0.601852          4.653023       5.254875
2    2017-03-01 00:00:00        0.880642          4.098880       3.218239
3    2017-04-01 00:00:00       -0.414137          2.601595       3.015732
4    2017-05-01 00:00:00        1.835914          3.808593       1.972679
..                   ...             ...               ...            ...
109  2026-02-01 00:00:00       48.732797         53.619473       4.886676
110  2026-03-01 00:00:00       50.786784         54.887756       4.100972
111  2026-04-01 00:00:00       52.646478         53.886603       1.240125
112  2026-05-01 00:00:00       50.112072         54.482149       4.370076
113  2026-06-01 00:00:00       46.559401         53.396786       6.837386

[114 rows x 4 columns]


QUESTION 3 : which flat type gives the best value (price per sqm)?

ANSWER 3 : Executive


In [9]:
df_flat = pd.read_sql_query('''
                       SELECT flat_type,
                            AVG(resale_price) AS avg_resale_price,
                            AVG(floor_area_sqm) AS avg_floor_area_sqm,
                            ROUND(AVG(resale_price) / AVG(floor_area_sqm),0) AS price_per_sqm
                       FROM hdb_resale
                       GROUP BY flat_type
                       ORDER BY price_per_sqm
                       ''', conn)
df_flat[["avg_resale_price", "avg_floor_area_sqm", "price_per_sqm"]] = df_flat[["avg_resale_price", "avg_floor_area_sqm", "price_per_sqm"]].round(0).astype(int)
print(df_flat)

          flat_type  avg_resale_price  avg_floor_area_sqm  price_per_sqm
0         EXECUTIVE            738622                 145           5100
1  MULTI-GENERATION            862626                 161           5349
2            5 ROOM            631017                 118           5360
3            3 ROOM            376443                  68           5525
4            4 ROOM            535814                  95           5639
5            2 ROOM            305124                  46           6675
6            1 ROOM            214371                  31           6915


QUESTION 4 : how did prices change during COVID (2020–2021) vs post-COVID?

ANSWER 4 : increase 24.70%



In [48]:
df_resale_price_COVID = pd.read_sql_query('''
                                              SELECT ROUND(AVG(resale_price), 0) AS avg_resale_price
                                              FROM hdb_resale
                                              WHERE month BETWEEN '2020-01-01' AND '2021-12-01'
                                              ''', conn)
print(df_resale_price_COVID)

   avg_resale_price
0          482821.0


In [49]:
df_resale_price_aft_COVID = pd.read_sql_query('''
                                              SELECT ROUND(AVG(resale_price), 0) AS avg_resale_price
                                              FROM hdb_resale
                                              WHERE month > '2021-12-31'
                                              ''', conn)
print(df_resale_price_aft_COVID)

   avg_resale_price
0          602089.0


In [50]:
resale_price_COVID = df_resale_price_COVID["avg_resale_price"].iloc[0]
resale_price_aft_COVID = df_resale_price_aft_COVID["avg_resale_price"].iloc[0]
pct_change = (resale_price_aft_COVID - resale_price_COVID) / resale_price_COVID * 100
print(f"Resale Price during COVID : ${resale_price_COVID:,.0f}")
print(f"Resale Price after COVID : ${resale_price_aft_COVID:,.0f}")
print(f"Price change : {pct_change:.2f}%")



Resale Price during COVID : $482,821
Resale Price after COVID : $602,089
Price change : 24.70%


QUESTION 5 : how has MRT ridership trended over the same period as housing prices?
ANSWER 5 : housing resale prices and mrt ridership dropped during COVID, then both recovered strongly post-2022.

In [42]:
pd.read_sql_query("SELECT * FROM mrt_ridership LIMIT 10", conn)

,_id,year,mode,ridership
0,1,1995,MRT,740000
1,2,1995,LRT,0
2,3,1995,Bus,3009000
3,4,1996,MRT,850000
4,5,1996,LRT,0
5,6,1996,Bus,3118000
6,7,1997,MRT,911000
7,8,1997,LRT,0
8,9,1997,Bus,3116000
9,10,1998,MRT,946000


In [47]:
df_ridership = pd.read_sql_query('''
                                 SELECT year,
                                 SUM(ridership) AS total_ridership
                                 FROM mrt_ridership
                                 GROUP BY year
                                 ORDER BY year
                                 ''', conn)
df_ridership["total_ridership"] = df_ridership["total_ridership"].apply(lambda x: f"{x:,}")
print(df_ridership.tail(10))

    year total_ridership
20  2015       6,914,000
21  2016       7,214,000
22  2017       7,264,000
23  2018       7,538,000
24  2019       7,691,000
25  2020       5,040,000
26  2021       5,259,000
27  2022       6,390,000
28  2023       7,192,000
29  2024       7,459,000


In [51]:
conn.close()